### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    pi.category,
    pi.sub_category,
    pi.cost_price
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Calculate cost per year 

In [ ]:
# 确保 order_date 是 datetime
df_order_completed_shipped["order_date"] = pd.to_datetime(df_order_completed_shipped["order_date"])

# 增加年份列
df_order_completed_shipped["order_year"] = df_order_completed_shipped["order_date"].dt.year

# 计算每行订单项的成本
df_order_completed_shipped["order_cost"] = (
    df_order_completed_shipped["cost_price"] * df_order_completed_shipped["quantity"]
)

# 按 order_id + order_year 汇总每张订单的总成本
df_cost_per_order = (
    df_order_completed_shipped
    .groupby(["order_id", "order_year"], as_index=False)["order_cost"]
    .sum()
)

df_cost_per_order

### Calculate `total_price_before_tax` per order 

In [ ]:
# Order表事先已计算好了`total_price_before_tax`了

### Calculate `gross profit` and `gross margin` per order 

In [ ]:
# 先取每张订单的收入（去重订单行）
df_revenue_per_order = (
    df_order_completed_shipped[["order_id", "order_year", "total_price_before_tax"]]
    .drop_duplicates(subset=["order_id","order_year"])
)

# 合并成本和收入
df_gross_per_order = df_cost_per_order.merge(
    df_revenue_per_order,
    on=["order_id", "order_year"],
    how="left"
)

# 计算毛利和毛利率
df_gross_per_order["gross_profit"] = (
    df_gross_per_order["total_price_before_tax"] - df_gross_per_order["order_cost"]
)
df_gross_per_order["gross_margin"] = (
    df_gross_per_order["gross_profit"] / df_gross_per_order["total_price_before_tax"].replace(0, pd.NA)
)

# 按年份汇总成本、收入、毛利
df_yearly = (
    df_gross_per_order
    .groupby("order_year", as_index=False)
    .agg(
        order_cost=("order_cost", "sum"),
        total_revenue=("total_price_before_tax", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

df_yearly["gross_margin"] = df_yearly["gross_profit"] / df_yearly["total_revenue"]

df_yearly

In [ ]:
# 关闭数据库连接
engine.dispose()